In [11]:
import csv
import time
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt


In [12]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# --- GPU performance tuning (safe no-ops on CPU) --------------------------- #
# cuDNN autotuner: picks the fastest RNN/LSTM/GRU kernel for our fixed
# (batch, seq_len, hidden_dim) shapes after a few warmup iterations.
torch.backends.cudnn.benchmark = True
# TF32 matmuls on Ampere/Ada tensor cores: ~free throughput gain, negligible
# accuracy impact for this task.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16  # Ada Lovelace runs bf16 at full tensor-core rate,
                            # and unlike fp16 it needs no GradScaler.


Using device: cuda


In [13]:
CSV_PATH = "data/shakespeare/shakespeare.csv"

SEQ_LEN = 100          # length of each training sequence (in characters)
BATCH_SIZE = 256 if USE_AMP else 128
EMBED_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.2
LR = 3e-3
MAX_EPOCHS = 20
PATIENCE = 3           # early stopping patience (epochs with no val-loss improvement)
TRAIN_FRAC = 0.9
MAX_CORPUS_CHARS = 2_000_000  # cap corpus size so training stays reasonably fast
GRAD_CLIP = 5.0


In [14]:
def load_corpus() -> str:
    """Load raw text from shakespeare.csv."""
    print(f"Loading corpus from {CSV_PATH} ...")
    text_col_candidates = {
        "line", "playerline", "text", "dialogue", "content", "text_data",
    }
    lines = []
    with open(CSV_PATH, "r", encoding="utf-8", errors="ignore") as f:
        reader = csv.reader(f)
        rows = list(reader)
    if not rows:
        raise ValueError(f"{CSV_PATH} is empty.")

    header = [h.strip().lower() for h in rows[0]]
    data_rows = rows[1:]
    col_idx = None
    for i, h in enumerate(header):
        if h in text_col_candidates:
            col_idx = i
            break

    if col_idx is not None:
        for row in data_rows:
            if len(row) > col_idx:
                lines.append(row[col_idx])
    else:
        # No recognizable header -> treat first column of every row
        # (including the "header" row, in case there wasn't one) as text.
        for row in rows:
            if row:
                lines.append(row[0])

    text = "\n".join(lines)
    if len(text.strip()) < 1000:
        print("Parsed corpus looked too small; falling back to raw file read.")
        with open(CSV_PATH, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()

    if len(text) > MAX_CORPUS_CHARS:
        text = text[:MAX_CORPUS_CHARS]

    print(f"Corpus length: {len(text):,} characters")
    return text


text = load_corpus()
text[:500]  # quick peek


Loading corpus from data/shakespeare/shakespeare.csv ...
Corpus length: 2,000,000 characters


'character_id\nTHE_TRAGEDY_OF_TITUS_ANDRONICUS_A_ROMAN\nTHE_TRAGEDY_OF_TITUS_ANDRONICUS_A_ROMAN\nTHE_TRAGEDY_OF_TITUS_ANDRONICUS_A_ROMAN\nTHE_TRAGEDY_OF_TITUS_ANDRONICUS_A_ROMAN\nTHE_TRAGEDY_OF_TITUS_ANDRONICUS_A_ROMAN\nTHE_TRAGEDY_OF_TITUS_ANDRONICUS_A_ROMAN\nTHE_TRAGEDY_OF_TITUS_ANDRONICUS_A_ROMAN\nTHE_TRAGEDY_OF_TITUS_ANDRONICUS_A_ROMAN\nTHE_TRAGEDY_OF_TITUS_ANDRONICUS_A_ROMAN\nTHE_TRAGEDY_OF_TITUS_ANDRONICUS_A_ROMAN\nTHE_TRAGEDY_OF_TITUS_ANDRONICUS_A_ROMAN\nTHE_TRAGEDY_OF_TITUS_ANDRONICUS_A_ROMAN\nTHE_TRA'

In [15]:
def build_vocab(text: str):
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos


stoi, itos = build_vocab(text)
vocab_size = len(stoi)
print(f"Vocabulary size: {vocab_size}")

encoded = np.array([stoi[ch] for ch in text], dtype=np.int64)

split_idx = int(len(encoded) * TRAIN_FRAC)
train_encoded = encoded[:split_idx]
val_encoded = encoded[split_idx:]

print(f"Train chars: {len(train_encoded):,} | Val chars: {len(val_encoded):,}")


Vocabulary size: 31
Train chars: 1,800,000 | Val chars: 200,000


In [16]:
class CharDataset(Dataset):
    """Produces (input_seq, target_seq) pairs of length SEQ_LEN, where the
    target is the input shifted by one character (next-character prediction)."""

    def __init__(self, encoded: np.ndarray, seq_len: int):
        self.encoded = encoded
        self.seq_len = seq_len
        self.num_sequences = max(0, (len(encoded) - 1) // seq_len)

    def __len__(self):
        return self.num_sequences

    def __getitem__(self, idx):
        start = idx * self.seq_len
        end = start + self.seq_len
        x = self.encoded[start:end]
        y = self.encoded[start + 1:end + 1]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


train_ds = CharDataset(train_encoded, SEQ_LEN)
val_ds = CharDataset(val_encoded, SEQ_LEN)

# pin_memory + non_blocking .to(device) lets the H2D copy overlap with
# compute. num_workers=0: __getitem__ here is just a numpy slice, so worker
# processes add IPC overhead rather than saving time -- keep loading
# in-process for this dataset.
loader_kwargs = dict(pin_memory=USE_AMP, num_workers=0, persistent_workers=False)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **loader_kwargs
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=True, **loader_kwargs
)

print(f"Train batches/epoch: {len(train_loader)} | Val batches/epoch: {len(val_loader)}")


Train batches/epoch: 70 | Val batches/epoch: 7


In [17]:
class CharRNNModel(nn.Module):
    """Configurable character-level language model backed by RNN, LSTM, or GRU."""

    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout, cell_type):
        super().__init__()
        self.cell_type = cell_type.upper()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        rnn_kwargs = dict(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        if self.cell_type == "RNN":
            self.rnn = nn.RNN(**rnn_kwargs, nonlinearity="tanh")
        elif self.cell_type == "LSTM":
            self.rnn = nn.LSTM(**rnn_kwargs)
        elif self.cell_type == "GRU":
            self.rnn = nn.GRU(**rnn_kwargs)
        else:
            raise ValueError(f"Unknown cell_type: {cell_type}")

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb = self.embedding(x)                  # (B, T, E)
        out, hidden = self.rnn(emb, hidden)       # (B, T, H)
        out = self.dropout(out)
        logits = self.fc(out)                     # (B, T, V)
        return logits, hidden


In [18]:
def run_epoch(model, loader, criterion, optimizer=None):
    """Runs one epoch. If optimizer is provided, trains; otherwise evaluates."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=USE_AMP)
            y = y.to(DEVICE, non_blocking=USE_AMP)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                logits, _ = model(x)                       # (B, T, V)
                loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()

            preds = logits.argmax(dim=-1)
            total_correct += (preds == y).sum().item()
            total_count += y.numel()
            total_loss += loss.item() * y.numel()

    avg_loss = total_loss / total_count
    accuracy = total_correct / total_count
    return avg_loss, accuracy


In [19]:

def train_model(cell_type):
    """Builds, trains (with early stopping), checkpoints, and reports one model."""
    print(f"\n{'=' * 60}\nTraining {cell_type}\n{'=' * 60}")

    model = CharRNNModel(
        vocab_size=vocab_size,
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        cell_type=cell_type,
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    # fused=True launches a single fused CUDA kernel for the Adam update
    # instead of one kernel per parameter tensor.
    try:
        optimizer = torch.optim.Adam(model.parameters(), lr=LR, fused=USE_AMP)
    except TypeError:
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_loss = float("inf")
    epochs_no_improve = 0
    checkpoint_path = f"best_{cell_type.lower()}.pt"

    for epoch in range(1, MAX_EPOCHS + 1):
        t0 = time.time()
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        elapsed = time.time() - t0

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(
            f"[{cell_type}] Epoch {epoch:2d}/{MAX_EPOCHS} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
            f"{elapsed:.1f}s"
        )

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "cell_type": cell_type,
                    "vocab_size": vocab_size,
                    "config": {
                        "embed_dim": EMBED_DIM,
                        "hidden_dim": HIDDEN_DIM,
                        "num_layers": NUM_LAYERS,
                        "dropout": DROPOUT,
                    },
                },
                checkpoint_path,
            )
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"[{cell_type}] Early stopping at epoch {epoch} "
                      f"(no val improvement for {PATIENCE} epochs).")
                break

    # Reload best checkpoint before final evaluation for reporting.
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    final_train_loss, final_train_acc = run_epoch(model, train_loader, criterion, optimizer=None)
    final_val_loss, final_val_acc = run_epoch(model, val_loader, criterion, optimizer=None)

    results = {
        "model": cell_type,
        "train_loss": final_train_loss,
        "val_loss": final_val_loss,
        "train_acc": final_train_acc,
        "val_acc": final_val_acc,
    }
    return model, history, results


Train each architecture

In [20]:
histories = {}
all_results = []


RNN

In [21]:
rnn_model, history, results = train_model("RNN")
histories["RNN"] = history
all_results.append(results)



Training RNN
[RNN] Epoch  1/20 | train_loss=0.2954 train_acc=0.9295 | val_loss=0.0214 val_acc=0.9919 | 0.8s
[RNN] Epoch  2/20 | train_loss=0.0239 train_acc=0.9921 | val_loss=0.0195 val_acc=0.9921 | 0.5s
[RNN] Epoch  3/20 | train_loss=0.0216 train_acc=0.9923 | val_loss=0.0191 val_acc=0.9921 | 0.5s
[RNN] Epoch  4/20 | train_loss=0.0206 train_acc=0.9924 | val_loss=0.0187 val_acc=0.9919 | 0.6s
[RNN] Epoch  5/20 | train_loss=0.0204 train_acc=0.9924 | val_loss=0.0191 val_acc=0.9919 | 0.6s
[RNN] Epoch  6/20 | train_loss=0.0201 train_acc=0.9924 | val_loss=0.0192 val_acc=0.9917 | 0.6s
[RNN] Epoch  7/20 | train_loss=0.0201 train_acc=0.9924 | val_loss=0.0196 val_acc=0.9923 | 0.6s
[RNN] Early stopping at epoch 7 (no val improvement for 3 epochs).


LSTM


In [22]:
lstm_model, history, results = train_model("LSTM")
histories["LSTM"] = history
all_results.append(results)



Training LSTM
[LSTM] Epoch  1/20 | train_loss=0.8280 train_acc=0.7843 | val_loss=0.0300 val_acc=0.9915 | 1.9s
[LSTM] Epoch  2/20 | train_loss=0.0300 train_acc=0.9921 | val_loss=0.0223 val_acc=0.9919 | 1.9s
[LSTM] Epoch  3/20 | train_loss=0.0234 train_acc=0.9927 | val_loss=0.0202 val_acc=0.9923 | 1.8s
[LSTM] Epoch  4/20 | train_loss=0.0215 train_acc=0.9927 | val_loss=0.0199 val_acc=0.9923 | 1.7s
[LSTM] Epoch  5/20 | train_loss=0.0205 train_acc=0.9927 | val_loss=0.0196 val_acc=0.9921 | 1.6s
[LSTM] Epoch  6/20 | train_loss=0.0198 train_acc=0.9927 | val_loss=0.0189 val_acc=0.9921 | 1.6s
[LSTM] Epoch  7/20 | train_loss=0.0195 train_acc=0.9927 | val_loss=0.0192 val_acc=0.9921 | 1.6s
[LSTM] Epoch  8/20 | train_loss=0.0193 train_acc=0.9927 | val_loss=0.0189 val_acc=0.9919 | 1.6s
[LSTM] Epoch  9/20 | train_loss=0.0190 train_acc=0.9927 | val_loss=0.0190 val_acc=0.9928 | 1.7s
[LSTM] Early stopping at epoch 9 (no val improvement for 3 epochs).


GRU

In [23]:
gru_model, history, results = train_model("GRU")
histories["GRU"] = history
all_results.append(results)



Training GRU
[GRU] Epoch  1/20 | train_loss=0.4983 train_acc=0.8698 | val_loss=0.0239 val_acc=0.9921 | 1.9s
[GRU] Epoch  2/20 | train_loss=0.0239 train_acc=0.9923 | val_loss=0.0206 val_acc=0.9913 | 1.8s
[GRU] Epoch  3/20 | train_loss=0.0207 train_acc=0.9926 | val_loss=0.0188 val_acc=0.9921 | 1.9s
[GRU] Epoch  4/20 | train_loss=0.0201 train_acc=0.9926 | val_loss=0.0190 val_acc=0.9923 | 1.7s
[GRU] Epoch  5/20 | train_loss=0.0196 train_acc=0.9926 | val_loss=0.0183 val_acc=0.9928 | 1.7s
[GRU] Epoch  6/20 | train_loss=0.0193 train_acc=0.9926 | val_loss=0.0187 val_acc=0.9923 | 1.7s
[GRU] Epoch  7/20 | train_loss=0.0191 train_acc=0.9926 | val_loss=0.0189 val_acc=0.9921 | 1.7s
[GRU] Epoch  8/20 | train_loss=0.0190 train_acc=0.9926 | val_loss=0.0179 val_acc=0.9926 | 1.7s
[GRU] Epoch  9/20 | train_loss=0.0188 train_acc=0.9926 | val_loss=0.0191 val_acc=0.9921 | 1.7s
[GRU] Epoch 10/20 | train_loss=0.0188 train_acc=0.9926 | val_loss=0.0184 val_acc=0.9926 | 1.9s
[GRU] Epoch 11/20 | train_loss=0.018

Comparison table

In [24]:
def print_comparison_table(all_results):
    headers = ["Model", "Train Loss", "Val Loss", "Train Acc", "Val Acc"]
    rows = [
        [
            r["model"],
            f"{r['train_loss']:.4f}",
            f"{r['val_loss']:.4f}",
            f"{r['train_acc']:.4f}",
            f"{r['val_acc']:.4f}",
        ]
        for r in all_results
    ]

    col_widths = [max(len(headers[i]), *(len(row[i]) for row in rows)) for i in range(len(headers))]

    def fmt_row(row):
        return " | ".join(cell.ljust(col_widths[i]) for i, cell in enumerate(row))

    print(fmt_row(headers))
    print("-+-".join("-" * w for w in col_widths))
    for row in rows:
        print(fmt_row(row))

    print("\nMarkdown table:\n")
    print("| " + " | ".join(headers) + " |")
    print("|" + "|".join(["---:"] * len(headers)) + "|")
    for row in rows:
        print("| " + " | ".join(row) + " |")


print_comparison_table(all_results)


Model | Train Loss | Val Loss | Train Acc | Val Acc
------+------------+----------+-----------+--------
RNN   | 0.0176     | 0.0187   | 0.9930    | 0.9919 
LSTM  | 0.0178     | 0.0189   | 0.9930    | 0.9921 
GRU   | 0.0174     | 0.0179   | 0.9930    | 0.9926 

Markdown table:

| Model | Train Loss | Val Loss | Train Acc | Val Acc |
|---:|---:|---:|---:|---:|
| RNN | 0.0176 | 0.0187 | 0.9930 | 0.9919 |
| LSTM | 0.0178 | 0.0189 | 0.9930 | 0.9921 |
| GRU | 0.0174 | 0.0179 | 0.9930 | 0.9926 |


Training curves

In [25]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, hist in histories.items():
    epochs = range(1, len(hist["train_loss"]) + 1)
    axes[0].plot(epochs, hist["train_loss"], linestyle="--", label=f"{name} train")
    axes[0].plot(epochs, hist["val_loss"], linestyle="-", label=f"{name} val")

    axes[1].plot(epochs, hist["train_acc"], linestyle="--", label=f"{name} train")
    axes[1].plot(epochs, hist["val_acc"], linestyle="-", label=f"{name} val")

axes[0].set_title("Loss over epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].set_title("Accuracy over epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Next-char accuracy")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

fig.tight_layout()
plt.show()


C:\Users\shehr\AppData\Local\Temp\ipykernel_26184\22505482.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
